# Практика: первый распознаватель и его честная оценка

Сервис просит прототип к пятнице: маленькая подвыборка, baseline, kNN, таблица опытов.

In [ ]:
from pathlib import Path
import pandas as pd


def find_digits_csv() -> Path:
    for p in (Path('digits.csv'), Path('../../data/digits.csv'), Path('../data/digits.csv')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError('digits.csv не найден — положите файл рядом с ноутбуком')


DIGITS_PATH = find_digits_csv()
df = pd.read_csv(DIGITS_PATH)
PIXELS = [c for c in df.columns if c.startswith('p')]

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt


def show_digit(row, title=''):
    """Одна строка таблицы -> картинка 8x8."""
    values = [int(v) for v in row[PIXELS]]
    grid = [values[i * 8:(i + 1) * 8] for i in range(8)]
    plt.imshow(grid, cmap='gray_r')
    plt.title(title)
    plt.axis('off')


## 1. Подвыборка 400 картинок

`sub = df.sample(400, random_state=0)`; разбейте на 300 обучающих и 100 проверочных (`test_size=100`, `random_state=0`, `stratify=sub['label']`) -> `X_tr`, `X_te`, `y_tr`, `y_te`.

**Зачем подвыборка:** на паре важно успеть прогнать десяток опытов, а не один.

In [ ]:
sub = None
X_tr = X_te = y_tr = y_te = None
assert sub is not None and len(sub) == 400
assert X_tr is not None and len(X_tr) == 300 and len(X_te) == 100
assert list(X_tr.columns) == PIXELS
print(len(X_tr), len(X_te))

## 2. Baseline на этой подвыборке

Самая частая цифра в обучающей части -> `top_train`. Если всегда отвечать ею, доля верных ответов на проверочной части -> `baseline_acc`.

**Это то число, которое обязан побить kNN.**

In [ ]:
top_train = None
baseline_acc = None
assert top_train is not None and int(top_train) in range(10)
assert baseline_acc is not None and 0.02 < float(baseline_acc) < 0.3
print(top_train, round(float(baseline_acc), 3))

## 3. Масштаб считаем только по обучающей части

Возьмите `mins = X_tr.min()`, `rng = (X_tr.max() - X_tr.min()).replace(0, 1)` и примените их к **обоим** наборам -> `tr_scaled`, `te_scaled`.

Максимум в `te_scaled` -> `te_max`. Он может оказаться больше 1 — это нормально.

В `LEAK_NOTE` ответьте: почему нельзя считать min/max по всей таблице сразу.

In [ ]:
tr_scaled = None
te_scaled = None
te_max = None
LEAK_NOTE = ''
assert tr_scaled is not None and te_scaled is not None
assert float(tr_scaled.max().max()) <= 1.0 + 1e-9
assert te_max is not None and len(LEAK_NOTE) > 60
print(round(float(te_max), 3), LEAK_NOTE)

## 4. Таблица опытов по числу соседей

Для `k` из 1, 3, 5, 7, 9 обучите kNN на `X_tr` и посчитайте точность на `X_te`. Соберите `results` — DataFrame со столбцами `k` и `accuracy` (постройте из списка списков).

Лучшая точность -> `best_acc`, соответствующее k -> `best_k`.

In [ ]:
results = None
best_acc = None
best_k = None
assert results is not None and len(results) == 5
assert list(results.columns) == ['k', 'accuracy']
assert best_acc is not None and float(best_acc) > float(baseline_acc) + 0.5
assert int(best_k) in (1, 3, 5, 7, 9)
print(results)

## 5. Сколько стоит один ответ

kNN сравнивает картинку со **всеми** обучающими. Замерьте `time.perf_counter()` вокруг `predict` для обучения на 300 картинках -> `t_small` и на 1200 -> `t_big` (вторую модель обучите на `df.iloc[:1200]`, проверяйте те же 100 картинок).

**Вопрос:** во сколько раз выросло время и почему.

In [ ]:
import time

t_small = None
t_big = None
assert t_small is not None and t_big is not None
assert float(t_small) > 0 and float(t_big) > 0
print(round(float(t_small), 4), round(float(t_big), 4))

## 6. На чём распознаватель ошибается

Обучите kNN с `best_k`, найдите номера проверочных картинок с неверным ответом -> `wrong_positions` (список позиций 0…99). Нарисуйте до трёх таких картинок и сохраните `figures/errors.png`.

В `ERROR_NOTE` — что общего у ошибок.

In [ ]:
from pathlib import Path as _P
_P('figures').mkdir(exist_ok=True)
wrong_positions = None
errors_png = None
ERROR_NOTE = ''
assert wrong_positions is not None and len(wrong_positions) >= 1
assert errors_png is not None and _P(errors_png).exists()
assert len(ERROR_NOTE) > 40
print(wrong_positions, ERROR_NOTE)

## 7. Эксперимент: пиксели, которые всегда пустые

Найдите пиксели, у которых во всей таблице одно и то же значение -> `const_pixels`. Обучите kNN (`best_k`) без них -> `acc_no_const`.

В `CONST_NOTE` объясните результат. **Готового ответа нет.**

In [ ]:
const_pixels = None
acc_no_const = None
CONST_NOTE = ''
assert const_pixels is not None and len(const_pixels) >= 1
assert acc_no_const is not None and float(acc_no_const) > 0.8
assert len(CONST_NOTE) > 40
print(const_pixels, round(float(acc_no_const), 4), CONST_NOTE)

## 8. Расширение: два признака вместо 64

Постройте два признака: `ink` (сумма яркости) и `n_dark` (сколько пикселей ярче 8). Обучите kNN (`best_k`) только на них -> `acc_two`.

В `FEATURES_NOTE` — почему потеря такая большая.

In [ ]:
acc_two = None
FEATURES_NOTE = ''
assert acc_two is not None
assert float(acc_two) < float(best_acc) - 0.3
assert len(FEATURES_NOTE) > 40
print(round(float(acc_two), 4), FEATURES_NOTE)